# Synthetic Data Generation Comparison: Faker vs SDV vs DataSynthesizer

This notebook takes a small input file containing **column names and allowed example values**, generates synthetic rows using three approaches, and then checks whether each generated output stayed within the provided examples.

## Input format

Create either `input_schema.csv` or `input_schema.xlsx` with these columns:

| column_name | examples |
|---|---|
| city | Delhi, Mumbai, Pune |
| status | ACTIVE, INACTIVE |
| amount | 100, 200, 500 |

The notebook treats every value in `examples` as an allowed domain value.  
The final metric flags generated values that are **outside** the examples.


In [ ]:
# Optional install cell
# Run this once if the libraries are not installed in your environment.
# DataSynthesizer can be sensitive to Python/library versions; if it fails, keep Faker and SDV tests running.

%pip install -q pandas numpy openpyxl faker sdv sdmetrics DataSynthesizer


## 1. Configuration and input loading


In [2]:
import os
import re
import json
import math
import random
import warnings
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# -----------------------------
# User configuration
# -----------------------------
INPUT_FILE = "input_schema_sample.csv"   # can be .csv or .xlsx
OUTPUT_DIR = Path("synthetic_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

N_ROWS = 500
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# -----------------------------
# Helpers
# -----------------------------
def split_examples(value: Any) -> List[str]:
    """
    Splits example values from a cell.
    Supports comma, pipe, semicolon, or newline separated values.
    """
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = re.split(r"[,|;\n]+", text)
    return [p.strip() for p in parts if p.strip() != ""]

def infer_series_type(values: List[str]) -> str:
    """
    Very simple type inference.
    We keep allowed domains for validation exactly as strings, but convert
    seed data for SDV/DataSynthesizer where possible.
    """
    if not values:
        return "categorical"

    try:
        [int(v) for v in values]
        return "integer"
    except Exception:
        pass

    try:
        [float(v) for v in values]
        return "float"
    except Exception:
        pass

    parsed = pd.to_datetime(pd.Series(values), errors="coerce")
    if parsed.notna().all():
        return "datetime"

    return "categorical"

def cast_value(value: str, dtype: str):
    if dtype == "integer":
        return int(value)
    if dtype == "float":
        return float(value)
    if dtype == "datetime":
        return pd.to_datetime(value)
    return str(value)

def load_schema_examples(path: str):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Create input_schema.csv or input_schema.xlsx with columns: column_name, examples"
        )

    if path.suffix.lower() in [".xlsx", ".xls"]:
        schema_df = pd.read_excel(path)
    else:
        schema_df = pd.read_csv(path)

    required = {"column_name", "examples"}
    missing = required - set(schema_df.columns)
    if missing:
        raise ValueError(f"Input file missing columns: {missing}. Required columns are: {required}")

    allowed_raw = {}
    inferred_types = {}

    for _, row in schema_df.iterrows():
        col = str(row["column_name"]).strip()
        examples = split_examples(row["examples"])
        if not col or not examples:
            continue
        allowed_raw[col] = examples
        inferred_types[col] = infer_series_type(examples)

    if not allowed_raw:
        raise ValueError("No valid columns/examples found in input file.")

    return allowed_raw, inferred_types, schema_df

def build_seed_dataframe(allowed_raw, inferred_types, n_rows: int = 200) -> pd.DataFrame:
    """
    Builds a small 'real-like' training dataset from the examples only.
    Each value is randomly sampled from the allowed examples.
    """
    data = {}
    for col, values in allowed_raw.items():
        dtype = inferred_types[col]
        sampled = np.random.choice(values, size=n_rows, replace=True)
        data[col] = [cast_value(v, dtype) for v in sampled]
    return pd.DataFrame(data)

allowed_raw, inferred_types, schema_df = load_schema_examples(INPUT_FILE)
real_seed_df = build_seed_dataframe(allowed_raw, inferred_types, n_rows=max(200, N_ROWS))

print("Loaded schema:")
display(schema_df)

print("Inferred types:")
display(pd.DataFrame({"column_name": list(inferred_types.keys()), "inferred_type": list(inferred_types.values())}))

print("Seed training data sample:")
display(real_seed_df.head())


Loaded schema:


,column_name,examples
0,city,"Delhi, Mumbai, Pune, Bengaluru"
1,customer_type,"Retail, Corporate, Student"
2,payment_status,"SUCCESS, FAILED, PENDING"
3,amount,"100, 200, 500, 1000"


Inferred types:


,column_name,inferred_type
0,city,categorical
1,customer_type,categorical
2,payment_status,categorical
3,amount,integer


Seed training data sample:


,city,customer_type,payment_status,amount
0,Pune,Student,FAILED,1000
1,Bengaluru,Retail,SUCCESS,100
2,Delhi,Retail,SUCCESS,500
3,Pune,Student,FAILED,1000
4,Pune,Retail,FAILED,100


## 2. Create a sample input file if needed


In [3]:
# Run this cell only if you want a quick sample schema file.
sample_schema = pd.DataFrame({
    "column_name": ["city", "customer_type", "payment_status", "amount"],
    "examples": [
        "Delhi, Mumbai, Pune, Bengaluru",
        "Retail, Corporate, Student",
        "SUCCESS, FAILED, PENDING",
        "100, 200, 500, 1000"
    ]
})

sample_schema.to_csv("input_schema_sample.csv", index=False)
print("Created input_schema_sample.csv. Rename it to input_schema.csv or update INPUT_FILE.")
display(sample_schema)


Created input_schema_sample.csv. Rename it to input_schema.csv or update INPUT_FILE.


,column_name,examples
0,city,"Delhi, Mumbai, Pune, Bengaluru"
1,customer_type,"Retail, Corporate, Student"
2,payment_status,"SUCCESS, FAILED, PENDING"
3,amount,"100, 200, 500, 1000"


## 3. Faker generation

Faker usually generates realistic-looking values using providers like name, email, address, date, etc.  
For this test, because your requirement is **only from provided example values**, this cell uses Faker's random engine to sample from the allowed examples.


In [4]:
from faker import Faker

fake = Faker()
Faker.seed(RANDOM_SEED)

faker_rows = []
for _ in range(N_ROWS):
    row = {}
    for col, values in allowed_raw.items():
        dtype = inferred_types[col]
        chosen = fake.random_element(elements=values)
        row[col] = cast_value(chosen, dtype)
    faker_rows.append(row)

faker_df = pd.DataFrame(faker_rows)

faker_csv = OUTPUT_DIR / "faker_synthetic.csv"
faker_xlsx = OUTPUT_DIR / "faker_synthetic.xlsx"

faker_df.to_csv(faker_csv, index=False)
faker_df.to_excel(faker_xlsx, index=False)

print(f"Saved: {faker_csv}")
print(f"Saved: {faker_xlsx}")
display(faker_df.head())


Saved: synthetic_outputs\faker_synthetic.csv
Saved: synthetic_outputs\faker_synthetic.xlsx


,city,customer_type,payment_status,amount
0,Delhi,Retail,PENDING,500
1,Mumbai,Retail,SUCCESS,100
2,Delhi,Student,FAILED,100
3,Delhi,Retail,SUCCESS,200
4,Delhi,Student,SUCCESS,1000


## 4. SDV generation

SDV learns a statistical model from tabular data and samples new rows from that model.  
Here, the seed data is created only from your example values. SDV may still generate values outside the original examples, especially for numeric columns, so the metric section checks that.

In this notebook numeric columns are deliberately marked as categorical because your stated requirement is exact allowed-value generation from the examples.


In [5]:
try:
    from sdv.metadata import SingleTableMetadata
    from sdv.single_table import GaussianCopulaSynthesizer

    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(real_seed_df)

    # Treat all columns as categorical to reduce out-of-domain values.
    # This is intentional because the user's requirement is "from examples only".
    for col in real_seed_df.columns:
        metadata.update_column(column_name=col, sdtype="categorical")

    sdv_model = GaussianCopulaSynthesizer(metadata)
    sdv_model.fit(real_seed_df)
    sdv_df = sdv_model.sample(num_rows=N_ROWS)

    # Normalize generated values to comparable forms
    for col, dtype in inferred_types.items():
        if dtype == "integer":
            sdv_df[col] = pd.to_numeric(sdv_df[col], errors="coerce").round().astype("Int64")
        elif dtype == "float":
            sdv_df[col] = pd.to_numeric(sdv_df[col], errors="coerce")
        elif dtype == "datetime":
            sdv_df[col] = pd.to_datetime(sdv_df[col], errors="coerce")

    sdv_csv = OUTPUT_DIR / "sdv_synthetic.csv"
    sdv_xlsx = OUTPUT_DIR / "sdv_synthetic.xlsx"

    sdv_df.to_csv(sdv_csv, index=False)
    sdv_df.to_excel(sdv_xlsx, index=False)

    print(f"Saved: {sdv_csv}")
    print(f"Saved: {sdv_xlsx}")
    display(sdv_df.head())

except Exception as e:
    print("SDV generation failed.")
    print("Error:", repr(e))
    sdv_df = pd.DataFrame()


Saved: synthetic_outputs\sdv_synthetic.csv
Saved: synthetic_outputs\sdv_synthetic.xlsx


,city,customer_type,payment_status,amount
0,Delhi,Corporate,FAILED,200
1,Mumbai,Retail,FAILED,500
2,Delhi,Corporate,FAILED,1000
3,Mumbai,Retail,PENDING,500
4,Bengaluru,Corporate,SUCCESS,200


## 5. DataSynthesizer generation

DataSynthesizer learns distributions from a dataset and can generate synthetic records from those learned distributions.  
It is older than Faker/SDV and may be more sensitive to environment versions. This cell uses the independent attribute mode.


In [8]:
try:
    from DataSynthesizer.DataDescriber import DataDescriber
    from DataSynthesizer.DataGenerator import DataGenerator

    ds_input = OUTPUT_DIR / "datasynthesizer_seed.csv"
    description_file = OUTPUT_DIR / "datasynthesizer_description.json"
    ds_output = OUTPUT_DIR / "datasynthesizer_synthetic.csv"
    ds_xlsx = OUTPUT_DIR / "datasynthesizer_synthetic.xlsx"

    # DataSynthesizer works best with CSV input.
    ds_seed_df = real_seed_df.copy()
    for col, dtype in inferred_types.items():
        if dtype == "datetime":
            ds_seed_df[col] = ds_seed_df[col].astype(str)
    ds_seed_df.to_csv(ds_input, index=False)

    describer = DataDescriber(category_threshold=1000)
    describer.describe_dataset_in_independent_attribute_mode(
        dataset_file=str(ds_input),
        epsilon=0,
        attribute_to_is_categorical={col: True for col in ds_seed_df.columns},
        attribute_to_is_candidate_key={col: False for col in ds_seed_df.columns}
    )
    describer.save_dataset_description_to_file(str(description_file))

    generator = DataGenerator()
    generator.generate_dataset_in_independent_mode(
        N_ROWS,
        str(description_file)
    )
    generator.save_synthetic_data(str(ds_output))

    datasynth_df = pd.read_csv(ds_output)

    for col, dtype in inferred_types.items():
        if col in datasynth_df.columns:
            if dtype == "integer":
                datasynth_df[col] = pd.to_numeric(datasynth_df[col], errors="coerce").round().astype("Int64")
            elif dtype == "float":
                datasynth_df[col] = pd.to_numeric(datasynth_df[col], errors="coerce")
            elif dtype == "datetime":
                datasynth_df[col] = pd.to_datetime(datasynth_df[col], errors="coerce")

    datasynth_df.to_csv(ds_output, index=False)
    datasynth_df.to_excel(ds_xlsx, index=False)

    print(f"Saved: {ds_output}")
    print(f"Saved: {ds_xlsx}")
    display(datasynth_df.head())

except Exception as e:
    print("DataSynthesizer generation failed.")
    print("Error:", repr(e))
    datasynth_df = pd.DataFrame()


Saved: synthetic_outputs\datasynthesizer_synthetic.csv
Saved: synthetic_outputs\datasynthesizer_synthetic.xlsx


,city,customer_type,payment_status,amount
0,Mumbai,Corporate,PENDING,200
1,Mumbai,Retail,FAILED,1000
2,Mumbai,Retail,PENDING,100
3,Mumbai,Student,SUCCESS,200
4,Delhi,Retail,FAILED,1000


## 6. Metric: check if generated data stayed within examples

This metric can run against each generated CSV/XLSX independently.  
It answers:

1. Did the output contain expected columns?
2. Did it create any extra columns?
3. For each column, how many generated values are outside the allowed examples?
4. Which invalid values appeared?
5. What percentage of rows are valid?


In [9]:
def normalize_for_compare(value: Any, dtype: str) -> str:
    if pd.isna(value):
        return "<NULL>"
    if dtype == "integer":
        try:
            return str(int(float(value)))
        except Exception:
            return str(value).strip()
    if dtype == "float":
        try:
            f = float(value)
            return str(int(f)) if f.is_integer() else str(f)
        except Exception:
            return str(value).strip()
    if dtype == "datetime":
        try:
            return pd.to_datetime(value).strftime("%Y-%m-%d")
        except Exception:
            return str(value).strip()
    return str(value).strip()

def read_generated_file(path):
    path = Path(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)

def validate_generated_file(generated_file, allowed_raw, inferred_types, save_report=True):
    generated_file = Path(generated_file)
    df = read_generated_file(generated_file)

    expected_cols = set(allowed_raw.keys())
    actual_cols = set(df.columns)

    missing_cols = sorted(expected_cols - actual_cols)
    extra_cols = sorted(actual_cols - expected_cols)

    summary_rows = []
    invalid_detail_rows = []

    for col in sorted(expected_cols):
        dtype = inferred_types[col]
        allowed_norm = {normalize_for_compare(v, dtype) for v in allowed_raw[col]}

        if col not in df.columns:
            summary_rows.append({
                "file": generated_file.name,
                "column": col,
                "status": "MISSING_COLUMN",
                "rows": len(df),
                "valid_count": 0,
                "invalid_count": len(df),
                "null_count": None,
                "valid_pct": 0.0,
                "invalid_values": "COLUMN_NOT_FOUND"
            })
            continue

        generated_norm = df[col].apply(lambda x: normalize_for_compare(x, dtype))
        invalid_mask = ~generated_norm.isin(allowed_norm)
        null_mask = generated_norm.eq("<NULL>")
        invalid_values = sorted(generated_norm[invalid_mask].dropna().unique().tolist())

        summary_rows.append({
            "file": generated_file.name,
            "column": col,
            "status": "PASS" if invalid_mask.sum() == 0 else "FAIL",
            "rows": len(df),
            "valid_count": int((~invalid_mask).sum()),
            "invalid_count": int(invalid_mask.sum()),
            "null_count": int(null_mask.sum()),
            "valid_pct": round((~invalid_mask).mean() * 100, 2) if len(df) else 0.0,
            "invalid_values": ", ".join(map(str, invalid_values[:20]))
        })

        if invalid_mask.sum() > 0:
            details = df.loc[invalid_mask, [col]].copy()
            details["file"] = generated_file.name
            details["column"] = col
            details["generated_value"] = generated_norm[invalid_mask].values
            details = details[["file", "column", "generated_value"]].drop_duplicates()
            invalid_detail_rows.extend(details.to_dict("records"))

    if missing_cols or extra_cols:
        summary_rows.insert(0, {
            "file": generated_file.name,
            "column": "__FILE_STRUCTURE__",
            "status": "FAIL",
            "rows": len(df),
            "valid_count": None,
            "invalid_count": None,
            "null_count": None,
            "valid_pct": None,
            "invalid_values": f"missing_cols={missing_cols}; extra_cols={extra_cols}"
        })

    summary = pd.DataFrame(summary_rows)
    invalid_details = pd.DataFrame(invalid_detail_rows)

    if save_report:
        report_base = OUTPUT_DIR / f"metric_report_{generated_file.stem}"
        summary.to_csv(f"{report_base}_summary.csv", index=False)
        summary.to_excel(f"{report_base}_summary.xlsx", index=False)
        if not invalid_details.empty:
            invalid_details.to_csv(f"{report_base}_invalid_values.csv", index=False)
            invalid_details.to_excel(f"{report_base}_invalid_values.xlsx", index=False)

    return summary, invalid_details

def run_metric_for_all_outputs():
    output_files = [
        OUTPUT_DIR / "faker_synthetic.csv",
        OUTPUT_DIR / "sdv_synthetic.csv",
        OUTPUT_DIR / "datasynthesizer_synthetic.csv"
    ]

    all_summaries = []

    for file in output_files:
        if file.exists():
            print(f"Validating {file} ...")
            summary, invalid_details = validate_generated_file(file, allowed_raw, inferred_types)
            all_summaries.append(summary)
            display(summary)
            if not invalid_details.empty:
                print("Invalid values found:")
                display(invalid_details.head(50))
            else:
                print("No invalid values found.")
        else:
            print(f"Skipping missing file: {file}")

    if all_summaries:
        combined = pd.concat(all_summaries, ignore_index=True)
        combined.to_csv(OUTPUT_DIR / "combined_metric_summary.csv", index=False)
        combined.to_excel(OUTPUT_DIR / "combined_metric_summary.xlsx", index=False)
        return combined

    return pd.DataFrame()

combined_metric_summary = run_metric_for_all_outputs()


Validating synthetic_outputs\faker_synthetic.csv ...


,file,column,status,rows,valid_count,invalid_count,null_count,valid_pct,invalid_values
0,faker_synthetic.csv,amount,PASS,500,500,0,0,100.0,
1,faker_synthetic.csv,city,PASS,500,500,0,0,100.0,
2,faker_synthetic.csv,customer_type,PASS,500,500,0,0,100.0,
3,faker_synthetic.csv,payment_status,PASS,500,500,0,0,100.0,


No invalid values found.
Validating synthetic_outputs\sdv_synthetic.csv ...


,file,column,status,rows,valid_count,invalid_count,null_count,valid_pct,invalid_values
0,sdv_synthetic.csv,amount,PASS,500,500,0,0,100.0,
1,sdv_synthetic.csv,city,PASS,500,500,0,0,100.0,
2,sdv_synthetic.csv,customer_type,PASS,500,500,0,0,100.0,
3,sdv_synthetic.csv,payment_status,PASS,500,500,0,0,100.0,


No invalid values found.
Validating synthetic_outputs\datasynthesizer_synthetic.csv ...


,file,column,status,rows,valid_count,invalid_count,null_count,valid_pct,invalid_values
0,datasynthesizer_synthetic.csv,amount,PASS,500,500,0,0,100.0,
1,datasynthesizer_synthetic.csv,city,PASS,500,500,0,0,100.0,
2,datasynthesizer_synthetic.csv,customer_type,PASS,500,500,0,0,100.0,
3,datasynthesizer_synthetic.csv,payment_status,PASS,500,500,0,0,100.0,


No invalid values found.


## 7. Optional: compare statistical similarity using SDMetrics


In [10]:
# This section is optional.
# It compares generated data against the seed training data statistically.
# The domain validation above is still the main metric for your exact requirement.

try:
    from sdmetrics.reports.single_table import QualityReport
    from sdv.metadata import SingleTableMetadata

    def run_sdmetrics_quality_report(synthetic_df: pd.DataFrame, name: str):
        if synthetic_df is None or synthetic_df.empty:
            print(f"Skipping {name}: empty data")
            return None

        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(real_seed_df)

        report = QualityReport()
        report.generate(real_seed_df, synthetic_df[real_seed_df.columns], metadata.to_dict())

        print(f"{name} quality score:", report.get_score())
        details = report.get_details(property_name="Column Shapes")
        display(details)
        return report

    if "faker_df" in globals():
        faker_quality = run_sdmetrics_quality_report(faker_df, "Faker")
    if "sdv_df" in globals() and not sdv_df.empty:
        sdv_quality = run_sdmetrics_quality_report(sdv_df, "SDV")
    if "datasynth_df" in globals() and not datasynth_df.empty:
        datasynth_quality = run_sdmetrics_quality_report(datasynth_df, "DataSynthesizer")

except Exception as e:
    print("Optional SDMetrics report failed.")
    print("Error:", repr(e))


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 1591.46it/s]|
Column Shapes Score: 95.2%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 254.43it/s]|
Column Pair Trends Score: nan%

Overall Score (Average): 95.2%

Faker quality score: 0.952


,Column,Metric,Score
0,customer_type,TVComplement,0.964
1,payment_status,TVComplement,0.970
2,amount,TVComplement,0.922


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 1997.53it/s]|
Column Shapes Score: 95.47%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 826.19it/s]|
Column Pair Trends Score: nan%

Overall Score (Average): 95.47%

SDV quality score: 0.9546666666666667


,Column,Metric,Score
0,customer_type,TVComplement,0.980
1,payment_status,TVComplement,0.958
2,amount,TVComplement,0.926


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 2000.86it/s]|
Column Shapes Score: 97.8%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 922.13it/s]|
Column Pair Trends Score: nan%

Overall Score (Average): 97.8%

DataSynthesizer quality score: 0.9780000000000001


,Column,Metric,Score
0,customer_type,TVComplement,0.992
1,payment_status,TVComplement,0.970
2,amount,TVComplement,0.972


## Output files

After running the notebook, check the `synthetic_outputs/` folder.

Expected generated files:

- `faker_synthetic.csv`
- `sdv_synthetic.csv`
- `datasynthesizer_synthetic.csv`

Expected metric files:

- `metric_report_faker_synthetic_summary.csv`
- `metric_report_sdv_synthetic_summary.csv`
- `metric_report_datasynthesizer_synthetic_summary.csv`
- `combined_metric_summary.csv`
